In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/train_clean.csv')

In [ ]:
unit_mode = df['unit'].mode()[0]
df['unit'] = df['unit'].fillna(unit_mode)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sample_id      75000 non-null  int64  
 1   image_link     75000 non-null  object 
 2   price          75000 non-null  float64
 3   item_name      75000 non-null  object 
 4   pack_quantity  75000 non-null  int64  
 5   value          75000 non-null  float64
 6   unit           75000 non-null  object 
 7   bullet_points  75000 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 4.6+ MB


In [ ]:
df_t = pd.read_csv('/content/test_clean.csv')

In [ ]:
unit_mode = df_t['unit'].mode()[0]
df_t['unit'] = df_t['unit'].fillna(unit_mode)

In [ ]:
df_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sample_id      75000 non-null  int64  
 1   image_link     75000 non-null  object 
 2   item_name      75000 non-null  object 
 3   pack_quantity  75000 non-null  int64  
 4   value          75000 non-null  float64
 5   unit           75000 non-null  object 
 6   bullet_points  75000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 4.0+ MB


In [ ]:
import os
import time
import urllib.request
from pathlib import Path
from functools import partial
import multiprocessing
from tqdm import tqdm
from PIL import Image
from io import BytesIO
from time import time as timer

# --- Create placeholder image if download fails ---
def create_placeholder_image(image_save_path, size=(256, 256)):
    try:
        placeholder_image = Image.new('RGB', size, color='black')
        placeholder_image.save(image_save_path)
    except Exception:
        pass

# --- Function to download + resize a single image ---
def download_and_resize_image(image_link, savefolder, size=(256, 256), retries=3, delay=2, timeout=10):
    """Download a single image with retries, resize, and placeholder on failure."""
    if not isinstance(image_link, str) or not image_link.strip():
        return

    filename = Path(image_link).name.split("?")[0]  # clean filename
    image_save_path = os.path.join(savefolder, filename)

    # Skip if already exists
    if os.path.exists(image_save_path):
        return

    for _ in range(retries):
        try:
            # --- Download with timeout ---
            with urllib.request.urlopen(image_link, timeout=timeout) as response:
                img_bytes = response.read()

            # --- Open image from memory and resize ---
            img = Image.open(BytesIO(img_bytes)).convert("RGB")
            img = img.resize(size, Image.Resampling.LANCZOS)

            # --- Save resized image ---
            img.save(image_save_path, format="JPEG", quality=85)
            return

        except Exception:
            time.sleep(delay)

    # Create placeholder if all retries fail
    create_placeholder_image(image_save_path, size=size)

# --- Function to parallelize downloads ---
def download_images(image_links, download_folder, num_workers=100, size=(256, 256), allow_multiprocessing=True):
    """Download and resize multiple images in parallel."""
    os.makedirs(download_folder, exist_ok=True)

    # Partial for multiprocessing
    download_partial = partial(download_and_resize_image, savefolder=download_folder, size=size)

    if allow_multiprocessing:
        with multiprocessing.Pool(processes=num_workers) as pool:
            list(
                tqdm(
                    pool.imap_unordered(download_partial, image_links),
                    total=len(image_links),
                    desc=f"Downloading + resizing to {size}",
                    smoothing=0.05
                )
            )
    else:
        for link in tqdm(image_links, total=len(image_links), desc=f"Downloading + resizing to {size}"):
            download_and_resize_image(link, savefolder=download_folder, size=size)

# --- Main runner ---
def run_image_download(train_df, image_folder="train_product_images", num_workers=100, size=(256, 256)):
    image_links = df["image_link"].dropna().tolist()
    print(f"Starting download + resize of {len(image_links)} images...")

    start = timer()
    download_images(image_links, image_folder, num_workers=num_workers, size=size)
    end = timer()
    print(f" Download + resize complete in {end - start:.2f} seconds.")
    print(f"Images saved to: {image_folder}")


run_image_download(df, image_folder="train_product_images", num_workers=100, size=(256, 256))


Starting download + resize of 75000 images...


In [ ]:
from pathlib import Path

df['local_image_path'] = df['image_link'].apply(lambda x: os.path.join('train_product_images', Path(x).name.split("?")[0]))
df.to_csv('train_with_image_paths.csv', index=False)

In [ ]:
# SECTION 1: Imports & Configuration
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
#from catboost import CatBoostRegressor, Pool
import torch
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import re
import html

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = html.unescape(text)                  # remove HTML escapes
    text = re.sub(r'<[^>]+>', ' ', text)        # remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s.,]', ' ', text)  # keep alphanumerics and simple punct
    text = text.lower()                         # lowercase for consistency
    text = re.sub(r'\s+', ' ', text).strip()    # collapse spaces
    return text


In [ ]:
# Optional cleaning
df['item_name'] = df['item_name'].fillna('')
df['bullet_points'] = df['bullet_points'].fillna('')
df['unit'] = df['unit'].fillna('unknown')

# Combine textual fields for embedding
df['text_combined'] = df['item_name'] + ' ' + df['bullet_points']

In [ ]:
# Apply cleaning before embedding
df['item_name'] = df['item_name'].fillna('').apply(clean_text)
df['bullet_points'] = df['bullet_points'].fillna('').apply(clean_text)

# Combine
df['text_combined'] = df['item_name'] + ' ' + df['bullet_points']


In [ ]:
def preprocess_and_engineer(df):
    df = df.copy()

    # Ensure text fields are properly formatted
    df['item_name'] = df['item_name'].fillna("").astype(str)
    df['bullet_points'] = df['bullet_points'].fillna("").apply(
        lambda x: " ".join(x) if isinstance(x, list) else str(x)
    )
    df['unit'] = df['unit'].fillna("unknown").astype(str)
    df['ocr_text'] = df['ocr_text'].fillna("").astype(str)

    # Log-transform price
    df['price_log'] = np.log1p(df['price'])

    # Feature engineering: text statistics
    df['item_name_len'] = df['item_name'].apply(len)
    df['item_name_wordcount'] = df['item_name'].apply(lambda x: len(x.split()))
    df['bullet_len'] = df['bullet_points'].apply(len)
    df['bullet_wordcount'] = df['bullet_points'].apply(lambda x: len(x.split()))

    # OCR text features
    df['ocr_len'] = df['ocr_text'].apply(len)
    df['ocr_wordcount'] = df['ocr_text'].apply(lambda x: len(x.split()))

    # Interaction / ratio features
    df['value_per_pack'] = df['value'] / (df['pack_quantity'] + 1e-6)
    df['ocr_item_overlap'] = df.apply(lambda r: len(set(r['ocr_text'].split()) & set(r['item_name'].split())), axis=1)

    return df


In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
from tqdm import tqdm
import os

In [ ]:
image_col = 'local_image_path'

In [ ]:
df = df[df[image_col].notna() & df[image_col].apply(lambda x: isinstance(x, str) and os.path.exists(x))].reset_index(drop=True)
print(f"Valid image paths found: {len(df)}")

Valid image paths found: 75000


In [ ]:
from multiprocessing.dummy import Pool as ThreadPool

In [ ]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load BLIP
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
model.eval()

# Optional: half precision for faster inference
model.half()


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


BlipForConditionalGeneration(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-0

In [ ]:
import os
import cv2
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
def resize_image_cv2(img_path, save_folder, max_size=256):
    try:
        if not isinstance(img_path, str) or not os.path.exists(img_path):
            return None
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        scale = max_size / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        save_path = os.path.join(save_folder, os.path.basename(img_path))
        cv2.imwrite(save_path, resized)
        return save_path
    except:
        return None


In [ ]:
from concurrent.futures import ProcessPoolExecutor

In [ ]:
def resize_wrapper(img_path):
    return resize_image_cv2(img_path, resized_folder, MAX_SIZE)


In [ ]:
image_col = "local_image_path"
resized_folder = "train_resized_images"
os.makedirs(resized_folder, exist_ok=True)
MAX_SIZE = 256
NUM_WORKERS = 4  # adjust based on CPU cores

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:

    resized_paths = list(tqdm(
        executor.map(resize_wrapper, df[image_col]),
        total=len(df),
        desc="Resizing images",
        unit='img',
        leave=True
    ))

df['resized_image_path'] = resized_paths

Resizing images:  96%|█████████▌| 71658/75000 [43:00<01:35, 34.84img/s]

In [ ]:
BATCH_SIZE = 32  # adjust based on GPU memory
all_captions = []

# Process in batches
for start in range(0, len(df), BATCH_SIZE):
    batch_imgs = df['resized_image_path'].iloc[start:start+BATCH_SIZE].tolist()

    # Load and preprocess images
    pil_imgs = []
    for img_path in batch_imgs:
        try:
            pil_imgs.append(Image.open(img_path).convert("RGB"))
        except:
            pil_imgs.append(None)

    # Filter out None images
    valid_indices = [i for i, x in enumerate(pil_imgs) if x is not None]
    valid_imgs = [pil_imgs[i] for i in valid_indices]

    if valid_imgs:
        inputs = processor(images=valid_imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_length=50)
        captions = [processor.decode(o, skip_special_tokens=True) for o in out]

        # Insert back into batch
        batch_captions = [""] * len(pil_imgs)
        for idx, cap in zip(valid_indices, captions):
            batch_captions[idx] = cap
    else:
        batch_captions = [""] * len(pil_imgs)

    all_captions.extend(batch_captions)
    torch.cuda.empty_cache()
    print(f"Processed batch {start//BATCH_SIZE + 1}")

# Save to DataFrame
df['blip_caption'] = all_captions
df.to_csv('train_with_blip.csv', index=False)
print(" BLIP captions extraction complete")


In [ ]:
df = preprocess_and_engineer(df)

use_log_target = True
target = 'price_log' if use_log_target else 'price'

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9.,% ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['ocr_text'] = df['ocr_text'].fillna('').apply(clean_text)


In [ ]:
df['text_combined'] = (
    df['item_name'].fillna('') + ' ' +
    df['bullet_points'].fillna('') + ' ' +
    df['ocr_text'].fillna('')
)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
class PriceDataset(Dataset):
    def __init__(self, texts, prices, tokenizer, max_length=128):
        self.texts = texts
        self.prices = prices
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        price = self.prices[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(price, dtype=torch.float)
        return item


In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# For regression, set num_labels=1
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
texts = df['text_combined'].tolist()
prices = df['price'].values

train_texts, val_texts, train_prices, val_prices = train_test_split(
    texts, prices, test_size=0.2, random_state=12
)

train_dataset = PriceDataset(train_texts, train_prices, tokenizer)
val_dataset = PriceDataset(val_texts, val_prices, tokenizer)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.L1Loss()  # MAE for regression

epochs = 2  # 1-3 epochs usually enough
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = outputs.logits.squeeze(-1)
        loss = loss_fn(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} – Train MAE: {total_loss/len(train_loader):.4f}")


Epoch 1 – Train MAE: 14.1199
Epoch 2 – Train MAE: 11.5365


In [ ]:
model.eval()
embeddings = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model.distilbert(input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:,0,:].cpu().numpy()  # CLS token
        embeddings.append(cls_emb)

embeddings = np.vstack(embeddings)


In [ ]:
texts = df['text_combined'].fillna('').tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Add to DataFrame
for i in range(embeddings.shape[1]):
    df[f'embed_{i}'] = embeddings[:, i]


Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

In [ ]:
# Numeric features
num_features = ['pack_quantity', 'value']

# Categorical features
cat_features = ['unit']  # CatBoost handles object dtype natively

# Embedding features (from BERT / PCA)
embed_features = [col for col in df.columns if col.startswith('embed_') or col.startswith('pca_embed_')]

# All features to use
features = num_features + cat_features + embed_features

# Target
target = 'price'


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df[features],
    df[target],
    test_size=0.2,
    random_state=12
)


In [ ]:
from catboost import CatBoostRegressor, Pool

# CatBoost Pool allows native handling of categorical features
train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
val_pool = Pool(data=X_val, label=y_val, cat_features=cat_features)


In [ ]:
catboost_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function='MAE',
    eval_metric='MAE',
    task_type='GPU',          # Enable GPU
    devices='0',              # GPU ID
    random_seed=10,
    early_stopping_rounds=100,
    verbose=100
)

catboost_model.fit(
    train_pool,
    eval_set=val_pool
)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 16.5247604	test: 16.5482250	best: 16.5482250 (0)	total: 134ms	remaining: 4m 27s
100:	learn: 16.3597437	test: 16.3846448	best: 16.3846448 (100)	total: 4.12s	remaining: 1m 17s
200:	learn: 16.2153417	test: 16.2447875	best: 16.2447875 (200)	total: 6.85s	remaining: 1m 1s
300:	learn: 16.0868781	test: 16.1214000	best: 16.1214000 (300)	total: 9.53s	remaining: 53.8s
400:	learn: 15.9713031	test: 16.0126219	best: 16.0126219 (400)	total: 12.2s	remaining: 48.5s
500:	learn: 15.8656958	test: 15.9145437	best: 15.9145437 (500)	total: 15.3s	remaining: 45.7s
600:	learn: 15.7681656	test: 15.8249802	best: 15.8249802 (600)	total: 18.8s	remaining: 43.8s
700:	learn: 15.6795667	test: 15.7457927	best: 15.7457927 (700)	total: 21.5s	remaining: 39.8s
800:	learn: 15.5989031	test: 15.6752677	best: 15.6752677 (800)	total: 24.2s	remaining: 36.2s
900:	learn: 15.5248667	test: 15.6116771	best: 15.6116771 (900)	total: 26.9s	remaining: 32.8s
1000:	learn: 15.4560344	test: 15.5534167	best: 15.5534167 (1000)	total: 

In [ ]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# Predictions
preds = catboost_model.predict(X_val)

# MAE
mae = mean_absolute_error(y_val, preds)

# SMAPE
def smape(y_true, y_pred):
    return 100 / len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

smape_val = smape(y_val.values, preds)

print(f"MAE: {mae:.4f}")
print(f"SMAPE: {smape_val:.2f}%")


MAE: 15.1205
SMAPE: 65.93%
